# Naive Bayes Classifier (Missing-Value Support)

Custom categorical Naive Bayes that:
- Skips `NaN` entries during training and prediction
- Estimates `P(F_i,j = v_k | observed features)` — required by SEU scoring

In [16]:
import sys, os
sys.path.insert(0, os.path.join(os.path.abspath(".."), "src"))

import numpy as np

## Class Definition

In [17]:
class NaiveBayesCategorical:
    """Naive Bayes for categorical features with missing-value support."""

    def __init__(self, alpha=1.0):
        self.alpha = alpha  # Laplace smoothing

    def fit(self, X, y):
        """
        X : np.ndarray (n, d), may contain NaN
        y : np.ndarray (n,)   integer class labels
        """
        n, d = X.shape
        self.classes_ = np.unique(y)
        self.n_classes_ = len(self.classes_)
        self.d_ = d

        # Class priors
        class_counts = np.array([(y == c).sum() for c in self.classes_], dtype=float)
        self.log_prior_ = np.log(class_counts / class_counts.sum())

        # Find unique values per feature (ignoring NaN)
        self.feature_values_ = []
        for j in range(d):
            col = X[:, j]
            vals = np.unique(col[~np.isnan(col)]).astype(int)
            self.feature_values_.append(vals)

        # Count occurrences per (class, feature value) with Laplace smoothing
        self.log_likelihood_ = []
        for j in range(d):
            vals = self.feature_values_[j]
            val_to_idx = {v: i for i, v in enumerate(vals)}
            counts = np.zeros((self.n_classes_, len(vals)), dtype=float)
            col = X[:, j]
            observed = ~np.isnan(col)
            for i in range(n):
                if observed[i]:
                    ci = np.searchsorted(self.classes_, y[i])
                    v = int(col[i])
                    if v in val_to_idx:
                        counts[ci, val_to_idx[v]] += 1
            counts += self.alpha
            log_probs = np.log(counts / counts.sum(axis=1, keepdims=True))
            self.log_likelihood_.append((val_to_idx, log_probs))

        return self

## Prediction Methods

In [18]:
def predict_log_proba(self, X):
    """Returns log P(C=c | x) (unnormalized). Shape: (n, n_classes)"""
    n = X.shape[0]
    log_prob = np.tile(self.log_prior_, (n, 1))

    for j in range(self.d_):
        col = X[:, j]
        val_to_idx, log_probs = self.log_likelihood_[j]
        for i in range(n):
            if not np.isnan(col[i]):
                v = int(col[i])
                if v in val_to_idx:
                    log_prob[i] += log_probs[:, val_to_idx[v]]
    return log_prob

def predict_proba(self, X):
    log_prob = self.predict_log_proba(X)
    log_prob -= log_prob.max(axis=1, keepdims=True)  # numerical stability
    prob = np.exp(log_prob)
    prob /= prob.sum(axis=1, keepdims=True)
    return prob

def predict(self, X):
    prob = self.predict_proba(X)
    return self.classes_[np.argmax(prob, axis=1)]

NaiveBayesCategorical.predict_log_proba = predict_log_proba
NaiveBayesCategorical.predict_proba = predict_proba
NaiveBayesCategorical.predict = predict

## Feature Value Probability Estimator

Estimates $P(F_{i,j} = v_k \mid \text{observed features})$ — used by SEU to weight hypothetical acquisitions.

$$P(F=v \mid \text{obs}) = \sum_c P(F=v \mid C=c) \cdot P(C=c \mid \text{obs})$$

In [19]:
def feature_value_proba(self, X_row, feature_idx):
    """
    Estimate P(F_i,j = v_k) for all possible values v_k of feature j,
    given currently observed features in X_row.
    Returns: (values_array, probs_array)
    """
    # Posterior P(C | obs), excluding the queried feature
    log_prob = self.log_prior_.copy()
    for j in range(self.d_):
        if j == feature_idx:
            continue
        v = X_row[j]
        if not np.isnan(v):
            val_to_idx, log_probs = self.log_likelihood_[j]
            vi = int(v)
            if vi in val_to_idx:
                log_prob += log_probs[:, val_to_idx[vi]]

    log_prob -= log_prob.max()
    post_class = np.exp(log_prob)
    post_class /= post_class.sum()

    # P(F=v_k | obs) = sum_c P(F=v_k | C=c) * P(C=c | obs)
    vals = self.feature_values_[feature_idx]
    val_to_idx, log_probs = self.log_likelihood_[feature_idx]
    cond_probs = np.exp(log_probs)  # (n_classes, n_vals)
    marginal = cond_probs.T @ post_class  # (n_vals,)
    marginal /= marginal.sum()
    return vals, marginal

NaiveBayesCategorical.feature_value_proba = feature_value_proba

## Demo

In [20]:
import pandas as pd
from data_utils import load_breast_cancer_data, apply_mask

X_full, y = load_breast_cancer_data()

rng = np.random.default_rng(42)
X_masked_df, mask = apply_mask(pd.DataFrame(X_full), 0.10, rng)
X_masked = X_masked_df.values

# Train/val split
rng2 = np.random.default_rng(0)
idx = rng2.permutation(len(y))
split = int(0.8 * len(y))
train_idx, val_idx = idx[:split], idx[split:]

model = NaiveBayesCategorical()
model.fit(X_masked[train_idx], y[train_idx])
acc = (model.predict(X_masked[val_idx]) == y[val_idx]).mean()
print(f"Val accuracy on 10%-masked data: {acc:.4f}")

# Feature value probability for instance 0, feature 0
vals, probs = model.feature_value_proba(X_masked[0], feature_idx=0)
print(f"\nP(feature 0 = v | obs) for instance 0:")
for v, p in zip(vals, probs):
    print(f"  bin {v}: {p:.3f}")

Val accuracy on 10%-masked data: 0.9386

P(feature 0 = v | obs) for instance 0:
  bin 0: 0.006
  bin 1: 0.019
  bin 2: 0.013
  bin 3: 0.019
  bin 4: 0.031
  bin 5: 0.082
  bin 6: 0.101
  bin 7: 0.201
  bin 8: 0.264
  bin 9: 0.264
